In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymatgen.core import Structure
from pymatgen.analysis.diffraction.xrd import XRDCalculator


def simulate_xrd(
    cif_file,
    theta_min=5,
    theta_max=40,
    step=0.01,
    fwhm=0.1,
    wavelength=1.5406,
    output_file="simulated_xrd.csv"
):

    structure = Structure.from_file(cif_file)
    xrd = XRDCalculator(wavelength=wavelength)
    pattern = xrd.get_pattern(structure, two_theta_range=(theta_min, theta_max))

    two_theta = np.arange(theta_min, theta_max, step)
    intensity = np.zeros_like(two_theta)

    def gaussian_area_normalized(x, mu, fwhm):
        sigma = fwhm / (2*np.sqrt(2*np.log(2)))
        return (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-(x-mu)**2/(2*sigma**2))

    for peak_pos, peak_int in zip(pattern.x, pattern.y):
        g = gaussian_area_normalized(two_theta, peak_pos, fwhm)
        intensity += peak_int * g

    intensity = intensity / intensity.max() * 100

    df = pd.DataFrame({
        "2theta (deg)": two_theta,
        "Intensity (a.u.)": intensity
    })

    df.to_csv(output_file, index=False)

    plt.figure(figsize=(8,5))
    plt.plot(two_theta, intensity)
    plt.xlabel("2θ (degrees)")
    plt.ylabel("Intensity (a.u.)")
    plt.xlim(theta_min, theta_max)
    plt.tight_layout()
    plt.show()

    return df


if __name__ == "__main__":
    simulate_xrd("your_file.cif")